# ACE example

Download, load, and inspect ACE intervals using paths relative to `root_dir`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'functions').exists() and (candidate / 'download_data.py').exists():
            return candidate
    raise RuntimeError('Could not locate MHDTurbPy repository root from current working directory.')


root_dir = find_repo_root(Path.cwd().resolve())
os.chdir(root_dir)
print(f'root_dir = {root_dir}')

spedas_dir = root_dir / 'data'
spedas_dir.mkdir(parents=True, exist_ok=True)
os.environ['SPEDAS_DATA_DIR'] = str(spedas_dir)

sys.path.insert(0, str(root_dir / 'pyspedas'))
sys.path.insert(1, str(root_dir / 'functions'))

import pyspedas
import download_data as download
import general_functions as func
import interactive_figs


## Download interval

In [ ]:
cdf_lib_path = '/Applications/cdf/cdf/lib'  # update this path if needed
credentials = None

settings = {
    'Data_path': root_dir / 'data',
    'save_destination': root_dir / 'examples' / 'downloaded_intervals',
    'overwrite_files': 1,
    'save_all': True,
    'addit_time_around': 1,
    'gap_time_threshold': 5,
    'sc': 'ACE',
    'in_rtn': False,
    'ace_frame': 'GSE',
    'start_date': '2025-10-01 00:00',
    'end_date': '2025-10-10 00:00',
    'multiple_intervals': False,
    'duration': '4H',
    'Step': '210min',
    'part_resol': 100,
    'MAG_resol': 100,
    'upsample_low_freq_ts': False,
    'must_have_qtn': False,
    'Max_par_missing': 30,
    'Big_Gaps': {
        'E_big_gaps': 10,
        'SC_pot_big_gaps': 10,
        'Mag_big_gaps': 10,
        'Par_big_gaps': 10,
        'QTN_big_gaps': 10,
    },
}

vars_2_download = {
    'mag': {'datatype': 'h0'},
    'par': {'datatype': 'h0'},
}

generated_interval_list = download.generate_intervals(
    settings['start_date'],
    settings['end_date'],
    settings['Step'],
    settings['duration'],
)

results = download.main_function(
    generated_interval_list,
    settings,
    vars_2_download,
    cdf_lib_path,
    credentials,
    n_jobs=1,
)


## Load one downloaded interval

In [ ]:
sc = 'ACE'
which_int = 0
load_path = root_dir / 'examples' / 'downloaded_intervals' / sc

finnames = func.load_files(load_path, 'final.pkl')
gennames = func.load_files(load_path, 'general.pkl')
signames = func.load_files(load_path, 'sig_c_sig_r.pkl')

fin = pd.read_pickle(finnames[which_int])
gen = pd.read_pickle(gennames[which_int])
sig = pd.read_pickle(signames[which_int])

fin.head(2)


## Interactive interval picker

In [ ]:
plt.close('all')

interactive_figs.select_interval_for_downloaded_dataframes(
    sc='ACE',
    final_df=fin,
    general=gen,
    sig_df=sig,
    out_dir=root_dir / 'examples',
    panel_edits=None,
)
